In [1]:
from pathlib import Path
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import (
    roc_auc_score, f1_score, recall_score, precision_score,
    accuracy_score, confusion_matrix, roc_curve
)

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC

from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE

# Busca automática pelo CSV dentro de ~/Documentos/ProjetoDados
PROJETO_DIR = Path.home() / "Documentos" / "ProjetoDados"
arquivos_csv = list(PROJETO_DIR.rglob("*Telco-Customer-Churn*.csv"))

if not arquivos_csv:
    raise FileNotFoundError(f"Arquivo CSV não encontrado dentro de {PROJETO_DIR}. Arquivos existentes: {list(PROJETO_DIR.rglob('*'))}")

DATA_PATH = arquivos_csv[0]
ASSETS_DIR = PROJETO_DIR / "assets"
ASSETS_DIR.mkdir(parents=True, exist_ok=True)
print(f"Base carregada de: {DATA_PATH}")

# 1. Carregamento e Preparação Inicial
df = pd.read_csv(DATA_PATH)
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")

# Remoção justificada na Etapa 2: identificador + atributos sem significância estatística (p > 0.05)
df_model = df.drop(columns=["customerID", "gender", "PhoneService"])

X = df_model.drop(columns=["Churn"])
y = df_model["Churn"].map({"No": 0, "Yes": 1})

num_features = ["tenure", "MonthlyCharges", "TotalCharges"]
cat_features = [c for c in X.columns if c not in num_features]

# 2. Split Estratificado (80% Treino / 20% Teste)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print(f"Treino: {X_train.shape[0]} amostras | Teste: {X_test.shape[0]} amostras")

# 3. Pipeline de Pré-processamento
num_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

cat_transformer = Pipeline(steps=[
    ("onehot", OneHotEncoder(drop="first", handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(transformers=[
    ("num", num_transformer, num_features),
    ("cat", cat_transformer, cat_features)
])

# 4. Definição dos Modelos e Grades de Hiperparâmetros (Grid Search)
modelos_grid = {
    "Regressão Logística": (
        LogisticRegression(max_iter=1000, random_state=42),
        {"clf__C": [0.1, 1.0, 10.0]}
    ),
    "Árvore de Decisão": (
        DecisionTreeClassifier(random_state=42),
        {"clf__max_depth": [4, 6, 8, 10], "clf__min_samples_split": [10, 20]}
    ),
    "Random Forest": (
        RandomForestClassifier(random_state=42),
        {"clf__n_estimators": [100, 200], "clf__max_depth": [6, 10, None]}
    ),
    "KNN": (
        KNeighborsClassifier(),
        {"clf__n_neighbors": [5, 11, 21], "clf__weights": ["uniform", "distance"]}
    ),
    "SVM": (
        SVC(probability=True, random_state=42),
        {"clf__C": [0.5, 1.0, 5.0], "clf__kernel": ["rbf", "linear"]}
    )
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
resultados = []
curvas_roc = {}
matrizes_confusao = {}

# 5. Treinamento, Validação Cruzada e Avaliação no Conjunto de Teste
for nome, (estimator, param_grid) in modelos_grid.items():
    print(f"Treinando e otimizando: {nome}...")
    pipe = ImbPipeline(steps=[
        ("prep", preprocessor),
        ("smote", SMOTE(random_state=42)),
        ("clf", estimator)
    ])
    
    grid = GridSearchCV(pipe, param_grid=param_grid, cv=cv, scoring="f1", n_jobs=-1)
    grid.fit(X_train, y_train)
    
    best_model = grid.best_estimator_
    y_pred = best_model.predict(X_test)
    y_proba = best_model.predict_proba(X_test)[:, 1]
    
    tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()
    especificidade = tn / (tn + fp)
    
    resultados.append({
        "Modelo": nome,
        "Melhores Hiperparâmetros": str(grid.best_params_),
        "AUROC": round(roc_auc_score(y_test, y_proba), 4),
        "F1-Score": round(f1_score(y_test, y_pred), 4),
        "Sensibilidade (Recall)": round(recall_score(y_test, y_pred), 4),
        "Especificidade": round(especificidade, 4),
        "Precisão": round(precision_score(y_test, y_pred), 4),
        "Acurácia": round(accuracy_score(y_test, y_pred), 4)
    })
    
    curvas_roc[nome] = roc_curve(y_test, y_proba)
    matrizes_confusao[nome] = np.array([[tn, fp], [fn, tp]])

df_resultados = pd.DataFrame(resultados).sort_values(by="AUROC", ascending=False).reset_index(drop=True)
display(df_resultados)

# 6. Salvar resultados e artefatos para o App Streamlit e Relatório
df_resultados.to_csv(ASSETS_DIR / "resultados_modelos.csv", index=False)
joblib.dump({"curvas_roc": curvas_roc, "matrizes": matrizes_confusao}, ASSETS_DIR / "artefatos_avaliacao.pkl")
print(f"\nResultados e artefatos salvos com sucesso em: {ASSETS_DIR.resolve()}")

Base carregada de: /home/tharles-mazurkievicz/Documentos/ProjetoDados/data/Telco-Customer-Churn.csv
Treino: 5634 amostras | Teste: 1409 amostras
Treinando e otimizando: Regressão Logística...
Treinando e otimizando: Árvore de Decisão...
Treinando e otimizando: Random Forest...
Treinando e otimizando: KNN...
Treinando e otimizando: SVM...


/home/tharles-mazurkievicz/Documentos/ProjetoDados/venv/lib/python3.14/site-packages/sklearn/svm/_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
/home/tharles-mazurkievicz/Documentos/ProjetoDados/venv/lib/python3.14/site-packages/sklearn/svm/_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
/home/tharles-mazurkievicz/Documentos/ProjetoDados/venv/lib/python3.14/site-packages/sklearn/svm/_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
/home/tharles-mazurkievicz/Documentos/ProjetoDado

,Modelo,Melhores Hiperparâmetros,AUROC,F1-Score,Sensibilidade (Recall),Especificidade,Precisão,Acurácia
0,Random Forest,"{'clf__max_depth': 6, 'clf__n_estimators': 100}",0.8423,0.6305,0.7914,0.7401,0.5239,0.7537
1,Regressão Logística,{'clf__C': 0.1},0.8396,0.6138,0.7861,0.7198,0.5034,0.7374
2,SVM,"{'clf__C': 0.5, 'clf__kernel': 'rbf'}",0.8272,0.6147,0.7594,0.7430,0.5164,0.7473
3,KNN,"{'clf__n_neighbors': 21, 'clf__weights': 'unif...",0.8170,0.5914,0.8262,0.6502,0.4605,0.6969
4,Árvore de Decisão,"{'clf__max_depth': 4, 'clf__min_samples_split'...",0.8166,0.6144,0.7540,0.7469,0.5184,0.7488



Resultados e artefatos salvos com sucesso em: /home/tharles-mazurkievicz/Documentos/ProjetoDados/assets
